# Банковский ИИ-ассистент: Prompting → RAG → Fine-Tuning (LoRA)

Практическая работа по адаптации LLM на примере ответов о тарифах и продуктах банка.

**Репозиторий:** [llm-adaptation-banking-agent](https://github.com/pueraeternis/llm-adaptation-banking-agent)

## 0. Подготовка окружения (Colab)

In [ ]:
# Клонирование репозитория (в Colab; локально пропустите ячейку)
!git clone https://github.com/pueraeternis/llm-adaptation-banking-agent.git
%cd llm-adaptation-banking-agent

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from pathlib import Path

ROOT = Path(".").resolve()
DATA_DIR = ROOT / "data"
TARIFFS_PATH = DATA_DIR / "banking_tariffs.txt"
LORA_DATASET_PATH = DATA_DIR / "lora_dataset.jsonl"

assert TARIFFS_PATH.exists(), f"Не найден {TARIFFS_PATH}"
assert LORA_DATASET_PATH.exists(), f"Не найден {LORA_DATASET_PATH}"

print("Тарифы:", TARIFFS_PATH)
print("LoRA датасет:", LORA_DATASET_PATH)

## 1. Prompt Engineering

Базовый ответ модели с системным промптом без внешних знаний.

In [ ]:
SYSTEM_PROMPT = """Ты — вежливый банковский ассистент.
Отвечай только на вопросы о продуктах и тарифах банка.
Если не знаешь ответа — честно скажи, что уточнишь у специалиста.
Не придумывай цифры и условия."""

USER_QUESTION = "Сколько стоит обслуживание дебетовой карты «Стандарт»?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_QUESTION},
]

print("Сообщения для модели:")
for m in messages:
    print(f"  [{m['role']}]: {m['content'][:80]}..." if len(m['content']) > 80 else f"  [{m['role']}]: {m['content']}")

> **Задание:** подключите Hugging Face `pipeline` или API и сгенерируйте ответ. Сравните с ответом без системного промпта.

## 2. RAG (Retrieval-Augmented Generation)

Поиск релевантных фрагментов в `data/banking_tariffs.txt` и подстановка в контекст.

In [ ]:
tariffs_text = TARIFFS_PATH.read_text(encoding="utf-8")
chunks = [c.strip() for c in tariffs_text.split("\n## ") if c.strip()]

print(f"Чанков в базе знаний: {len(chunks)}")
print("\n--- Пример чанка ---\n")
print(chunks[0][:500])

In [ ]:
def simple_retrieve(query: str, chunks: list[str], top_k: int = 2) -> list[str]:
    """Наивный поиск по пересечению слов (замените на embeddings + FAISS)."""
    q_words = set(query.lower().split())
    scored = []
    for ch in chunks:
        score = len(q_words & set(ch.lower().split()))
        scored.append((score, ch))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [ch for _, ch in scored[:top_k]]

query = USER_QUESTION
context_chunks = simple_retrieve(query, chunks)
rag_context = "\n\n".join(context_chunks)

RAG_PROMPT = f"""Контекст из базы знаний банка:
{rag_context}

Вопрос клиента: {query}
Ответь, опираясь только на контекст."""

print(RAG_PROMPT[:800])

> **Задание:** реализуйте векторный поиск (`sentence-transformers` + `faiss-cpu`) и сравните качество ответов с наивным поиском.

## 3. Fine-Tuning (LoRA)

Дообучение на микро-датасете `data/lora_dataset.jsonl`.

In [ ]:
import json

records = []
with open(LORA_DATASET_PATH, encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print(f"Примеров в датасете: {len(records)}")
print("\nПример записи:")
print(json.dumps(records[0], ensure_ascii=False, indent=2))

In [ ]:
def format_alpaca(example: dict) -> str:
    return (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Input:\n{example['input']}\n\n"
        f"### Response:\n{example['output']}"
    )

print(format_alpaca(records[0]))

> **Задание:** настройте LoRA через `peft` + `trl.SFTTrainer` на компактной модели (например, `Qwen2.5-0.5B-Instruct`). Сохраните адаптер и протестируйте на тех же вопросах, что в RAG.

## 4. Сравнение методов

| Метод | Плюсы | Минусы |
|-------|-------|--------|
| Prompting | Быстро, без обучения | Модель может галлюцинировать |
| RAG | Актуальные знания без переобучения | Нужна качественная база и поиск |
| LoRA | Стиль и формат ответов «вшиты» | Нужны данные и GPU; знания устаревают |